---
# Chapter 16 — Learning From Experience

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 16: Learning From Experience |
| Central question | When is experience strong enough to justify changing how future work will be done? |
| Main concepts | Learning from experience, Outcome attribution, Procedure memory |
| Implementation | None (pending experiment) |
| Experiment | Experiment pending |
| Evidence status | Experimental / deferred — hypothesis only |
| Depends on | Chapter 12 (behavioural instrument), Chapter 11 (gates) |

---


## What this notebook demonstrates

> **Experiment pending.** No merged outcome-adaptation run exists. This notebook draws the memory/learning boundary with the repository's own gated machinery and states what would have to change for the boundary to move.

1. **Shows retained experience influencing the present** (a frozen ch12 remove/restore pair)
2. **Shows the learning machinery that refuses to engage by default** (associative `LearningConfig`, disabled)
3. **Runs the promotion discipline** that any policy change must survive (backtest: replay before promotion)

A retained experience and an evaluated policy change are different objects. The notebook keeps them separate.


## The chapter question

> **When does remembering become learning?**

**Memory** is retained past experience remaining capable of changing present behaviour. **Learning** is a change to the mechanism that determines how future situations will be processed, driven by evaluated experience. The second requires outcome attribution strong enough to be safe — which the book does not yet have.


## Concepts in this chapter


In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent


REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(16)
concepts = (meta.get("chapter", {}).get("concepts")
            or meta.get("concepts", []))
render_table([
    {"Concept ID": c["id"], "Name": c["name"], "Status": c["status"]}
    for c in concepts
], "Chapter 16 Concepts")

## Retained experience influencing the present (memory, not learning)

The frozen ch12 deltas already show the first half: restoring decisive memory moves `fix-store` by +0.667 and `release-blockers` by +0.5. Nothing about the mechanism changed between those runs — only the retained past. That is memory doing its job, and it is not learning.


In [ ]:
from notebooks.memory._support import load_frozen_run

deltas = load_frozen_run(
    "ch12-20260920T204414Z-behavior")["summary"]["deltas"]
for task in ("fix-store", "release-blockers", "review-arch"):
    row = deltas.get(f"{task}|BR", {})
    print(f"{task:18s} restore-decisive delta: {row.get('task_delta')} "
          f"(repeats: {row.get('repeats')})")
print("\nSame mechanism, different past, different behaviour: memory.")

## The learning machinery that stays off (for now)

Associative reinforcement exists in code and is gated on task outcomes, never on retrieval frequency — and it ships **disabled**. Strengthening an edge requires evidence confidence and provenance; frequency alone never qualifies. That gate is the boundary made executable: the day it turns on with replay discipline, the system starts learning.


In [ ]:
from associative_memory import LearningConfig

cfg = LearningConfig()
print("Learning enabled:", cfg.enabled)
print("Policy version:", cfg.policy_version)
print("Reinforce/weaken steps:", cfg.reinforce_step, cfg.weaken_step)
print("Min evidence confidence:", cfg.min_evidence_confidence)
print("Require provenance:", cfg.require_provenance)
print("\nFrequency never qualifies; outcomes gate everything. "
      "That is the memory/learning boundary in one config object.")

In [ ]:
# Promotion discipline: a candidate change replays over frozen tasks
# and promotes only on primary-metric gain with no gate breached.
# Here the frozen ch11 aggregates stand in as baseline vs candidate.
from derived_loops.backtest import PRIMARY, evaluate_candidate
from notebooks.memory._support import load_frozen_run

m = load_frozen_run(
    "ch11-20260920T165907Z-derived-loops")["metrics"]
verdict = evaluate_candidate(m["unconstrained"], m["staged"])
print("Primary metric:", PRIMARY)
print("Primary gain:", verdict["primary_gain"])
print("Breaches:", verdict["breaches"])
print("Promoted:", verdict["promoted"])
print("\nLearning would mean a change like this rewriting future "
      "processing. The replay ran; the rewrite did not happen here.")

## What this establishes

- **Memory ≠ learning**: same mechanism + different past (memory) vs changed mechanism from evaluated experience (learning)
- **The boundary is executable**: a disabled-by-default learner plus a replay-before-promotion gate
- **Outcome attribution is the missing strength**: the book's tasks do not yet support safe adaptation
- **Conservatism here is a cost judgement**, not an absence — Chapter 12's attribution ladder shows the effect is causal


## What this does NOT establish

- Any procedure extraction or precondition verification
- Continual learning across model changes
- Memory portability between readers
- That the frozen remove/restore deltas would survive at scale


In [ ]:
# TRY IT YOURSELF: flip the comparison — would the unconstrained
# listing promote over the staged gate? Read the verdict.
rev = evaluate_candidate(m["staged"], m["unconstrained"])
print("Primary gain:", rev["primary_gain"])
print("Breaches:", rev["breaches"])
print("Promoted:", rev["promoted"])

## Where this leads next

Chapter 17 draws the standing boundary: which memories may influence behaviour at all, and what suppresses the rest.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory\16-chapter.ipynb)
